In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8461.72it/s]


In [4]:
v1 = model.encode(q1)

In [5]:
v1.shape

(384,)

In [6]:
v2 = model.encode(q2)

In [7]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [8]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [9]:
v1.dot(dv)

np.float32(0.32332408)

In [10]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [11]:
v2.dot(dv)

np.float32(0.019730505)

In [12]:
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py


In [13]:
from vector_search.ingest import load_faq_data

documents = load_faq_data()

In [14]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [15]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [16]:
len(texts)

1350

In [17]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [18]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

100%|██████████| 27/27 [00:47<00:00,  1.77s/it]


1350

In [19]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [20]:
import numpy as np
X = np.array(vectors)

In [21]:
scores = X.dot(v1)

In [22]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629411))

In [23]:
documents[553]

{'course': 'llm-zoomcamp',
 'section': 'Module 1: RAG',
 'question': 'OpenAI: Error when running OpenAI responses.create command',
 'answer': 'You may receive the following error when running the OpenAI `responses.create` command due to insufficient credits in your OpenAI account:\n\n```\nOpenAI API Error: Insufficient credits\n```',
 'doc_id': 'f5df151c59'}

In [24]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

In [25]:
scores[top5]

array([0.7629411 , 0.7579373 , 0.71921325, 0.6536313 , 0.56009996],
      dtype=float32)

In [26]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629411
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579373
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.71921325
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions'

In [27]:
top5

array([  2, 625, 907, 538,   7])

In [28]:
top5 = np.argsort(-scores)[:5]

In [29]:
top5

array([  2, 625, 907, 538,   7])

In [30]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [31]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed an

In [32]:
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py


In [33]:
from dotenv import load_dotenv
from openai import OpenAI
import os

LLM_MODEL = "MiniMax-M2.7"

load_dotenv(override=True)

api_key = os.environ.get("OPENAI_API_KEY")
base_url = os.environ.get("OPENAI_BASE_URL")
if not api_key or not base_url:
    raise SystemExit(
        "Missing env vars. Set OPENAI_API_KEY and OPENAI_BASE_URL in .env."
    )

openai_client = OpenAI(api_key=api_key, base_url=base_url)

In [34]:
from vector_search.ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [35]:
from vector_search.rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    model=LLM_MODEL
)

In [36]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)


"Yes, you can still join! According to the course information:\n\n> Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.\n\nSo you can sign up and participate in the course. Just keep in mind that to earn a certificate, you'll need to complete and submit your Capstone project before the submission deadline."

In [37]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [38]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
    model=LLM_MODEL
)

In [39]:
query = 'I just found out about the program, can I still sign up?'
vector_assistant.rag(query)


'Yes, you can still join! However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [40]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [41]:
vs_index.fit(vectors, documents)


In [42]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [43]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [44]:
vs_index.close()